# Lab 04 · Exact search vs HNSW vs IVF

Module E • Core concepts + real `hnswlib` ANN experiment. Explore n_probe, recall@k, candidate counts and graph search. HNSW recall depends on settings and data.

**Workflow:** read → predict → run → edit the marked variable → rerun → answer the linked questions in `assessments/quiz.html`.  \n**No answer needs a screenshot.** Keep your notebook output visible while taking the quiz.

# Your task board · Run first, answer later

Use the teaching cells below to determine the answers. For an **EDIT** task, change the marked variable and rerun the specified cell. The hints and full answers are intentionally **open at the end of this notebook**. Try the questions first.

**E1 · NAT · after cell 04 · E1** — How many synthetic vectors does the index benchmark create?  
NAT: `_________`

**E2 · NAT · after cell 04 · E1** — How many dimensions does each benchmark vector have?  
NAT: `_________`

**E3 · NAT · after cell 04 · E1** — How many query vectors does the benchmark create?  
NAT: `_________`

**E4 · NAT · after cell 04 · E3** — How many IVF lists are built in the KMeans example?  
NAT: `_________`

**E5 · NAT · after cell 04 · E5** — In the tiny fixed IVF example, how many candidates when n_probe=1?  
NAT: `_________`

**E6 · NAT · after cell 04 · E5** — In the tiny fixed IVF example, how many candidates when n_probe=2?  
NAT: `_________`

**E7 · NAT · after cell 04 · E5** — In the tiny fixed IVF example, how many candidates when n_probe=3?  
NAT: `_________`

**E8 · MCQ · after cell 04 · E2** — Which setting controls how broadly HNSW searches at query time?
  - **A.** n_probe
  - **B.** ef_search
  - **C.** PCA components
  - **D.** temperature

---

**Assessment:** [Interactive self-check](../assessments/quiz.html?module=E) · [Full question sheet](../assessments/QUESTION_SHEET.md). You can learn without grading; 70% unlocks the next quiz on the portal.

**Official documentation:** [hnswlib official](https://github.com/nmslib/hnswlib) · [HNSW parameters](https://github.com/nmslib/hnswlib/blob/master/ALGO_PARAMS.md) · [Faiss: faster search](https://github.com/facebookresearch/faiss/wiki/Faster-search).

In [ ]:
from pathlib import Path
import numpy as np
from IPython.display import display

def find_workshop_root():
    candidates = [Path.cwd(), *Path.cwd().parents, Path('/content/embedding_workshop')]
    for p in candidates:
        if (p / 'data' / 'field_notes.md').is_file():
            return p
    raise FileNotFoundError('Cannot find data/field_notes.md. Unzip the complete workshop, not just one notebook. Check README.md.')
ROOT = find_workshop_root()
print('Workshop root:', ROOT)


## E0 · Install the real HNSW implementation
HNSW is a **multi-layer navigable graph**; IVF is a **coarse clustering / inverted-list** approach. A distance metric decides what *near* means; an index decides how to find neighbors faster.

In [ ]:
%pip -q install 'hnswlib>=0.8,<1.0' 

## E1 · Exact baseline over a reproducible synthetic dataset
This is **synthetic geometry** for index evaluation, not model-learned semantic embeddings.

In [ ]:
import hnswlib
from sklearn.cluster import KMeans
rng=np.random.default_rng(42)
N,D=480,8
centers=np.array([[0]*D,[4]+[0]*(D-1),[0,4]+[0]*(D-2)],dtype=float)
X=np.vstack([rng.normal(loc=centers[i],scale=.7,size=(N//3,D)) for i in range(3)]).astype('float32')
queries=X[[7,169,334]]+.06
K=5
sqdist=lambda a,b: np.sum((a-b)**2,axis=-1)
exact=np.argsort(sqdist(queries[:,None,:],X[None,:,:]),axis=1)[:,:K]
print('dataset_shape =',X.shape)
print('query_shape =',queries.shape)
print('top_k =',K)
print('first_exact_neighbors =',exact[0].tolist())

## E2 · HNSW real index
✏️ Change **EF_SEARCH = 10 → 100** and rerun. Larger ef_search *often* improves recall with more search work; no universal exact runtime guarantee.

In [ ]:
index=hnswlib.Index(space='l2',dim=D)
index.init_index(max_elements=N,ef_construction=100,M=12,random_seed=42)
index.add_items(X,np.arange(N))
EF_SEARCH=10 # ✏️ CHANGE TO 100
index.set_ef(EF_SEARCH)
hnsw_ids,hnsw_dist=index.knn_query(queries,k=K)
recall=lambda result: sum(len(set(exact[i])&set(result[i])) for i in range(len(exact)))/(len(exact)*K)
print('ef_search =',EF_SEARCH)
print('hnsw_first_neighbors =',hnsw_ids[0].tolist())
print('recall_at_5 =',round(recall(hnsw_ids),3))

## E3 · IVF: first train centroids, then create inverted lists
Here we implement small IVF **exactly** so you can inspect its lists. HNSW and IVF are alternative index structures; Chroma local is normally HNSW, not IVF.

In [ ]:
N_LISTS=12
km=KMeans(n_clusters=N_LISTS,random_state=42,n_init=10).fit(X)
assignments=km.labels_
lists={j:np.flatnonzero(assignments==j) for j in range(N_LISTS)}
print('number_of_lists =',len(lists))
print('sum_list_sizes =',sum(len(v) for v in lists.values()))
print('smallest_and_largest_list =',min(map(len,lists.values())),max(map(len,lists.values())))

## E4 · Set n_probe and inspect candidates
✏️ `N_PROBE = 1`, then `3`, then `12`. When every list is probed, candidate search equals exact search (assuming the same distance and no extra compression).

In [ ]:
def ivf_query(query, n_probe=1, k=5):
    probe=np.argsort(sqdist(km.cluster_centers_,query))[:n_probe]
    candidates=np.concatenate([lists[int(i)] for i in probe])
    result=candidates[np.argsort(sqdist(X[candidates],query))[:k]]
    return result, int(len(candidates)),probe.tolist()

N_PROBE=1 # ✏️ CHANGE TO 3, THEN 12
ivf_results=[ivf_query(q,n_probe=N_PROBE,k=K) for q in queries]
ivf_ids=np.stack([r[0] for r in ivf_results])
print('n_probe =',N_PROBE)
print('first_query_lists =',ivf_results[0][2])
print('first_query_candidates =',ivf_results[0][1])
print('ivf_recall_at_5 =',round(recall(ivf_ids),3))
if N_PROBE==N_LISTS:
    assert np.array_equal(ivf_ids,exact), 'Full-probe IVF should match exact neighbors here.' 

## E5 · A tiny, completely deterministic IVF worked example
Three fixed centroids and six labeled points. Unlike trained KMeans, every NAT in this cell has a stable numerical answer.

In [ ]:
tiny_points=np.array([[0.,0.],[1.,0.],[0.,1.],[5.,0.],[6.,0.],[0.,5.]])
tiny_centers=np.array([[0.,0.],[5.,0.],[0.,5.]])
tiny_labels=np.argmin(((tiny_points[:,None,:]-tiny_centers[None,:,:])**2).sum(axis=2),axis=1)
tiny_query=np.array([.2,.1])
centroid_order=np.argsort(((tiny_centers-tiny_query)**2).sum(axis=1))
print('tiny_assignments =',tiny_labels.tolist())
for nprobe in (1,2,3):
    chosen=centroid_order[:nprobe]
    candidate_ids=np.flatnonzero(np.isin(tiny_labels,chosen))
    print('n_probe =',nprobe,'chosen_lists =',chosen.tolist(),'candidate_ids =',candidate_ids.tolist(),'candidate_count =',len(candidate_ids))

### Finish Module E
Answer **E1–E8**. Compare IVF `n_probe=1` versus full-probe and HNSW `ef_search=10/100` in your own words.

## Hints · Use these only after attempting the tasks

- **E1:** Inspect the constant `N` in E1.
- **E2:** Inspect `D` and the shape of X.
- **E3:** Look at how `queries` is sliced and offset.
- **E4:** `N_LISTS` controls the number of centroids / inverted lists.
- **E5:** Find which records are assigned to the nearest fixed centroid.
- **E6:** Count records in both closest fixed clusters.
- **E7:** Probing all lists means every point becomes a candidate.
- **E8:** Find the `index.set_ef(...)` call.

These are learning hints, not the solutions. Scroll down for the fully open answer key.


## Open answers · Full worked key and explanations

The answers below refer to the **supplied dataset and stated default/EDIT settings**. If you changed additional inputs, your results may differ. Compare the experiment and formula—not just the number.

**E1: 480**  
N=480, divided across three seeded clusters.

**E2: 8**  
D=8.

**E3: 3**  
The code selects three records and offsets them.

**E4: 12**  
N_LISTS=12.

**E5: 3**  
The first centroid has points [0,1,2].

**E6: 5**  
Two closest centroids collectively own 3+2 points.

**E7: 6**  
Every list is selected.

**E8: B — ef_search**  
ef_search is a query-time HNSW search-width control.

### Extension challenge — Interview explanation

**Challenge:** Your HNSW index seems fast at 480 vectors. How would you run a useful latency/recall benchmark at 1 million vectors?

**Self-check:** Explain the trade-off aloud in 45 seconds using a concrete example.
